In [1]:
%pip install nltk

Note: you may need to restart the kernel to use updated packages.


In [9]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import nltk

# Download necessary NLTK data
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\solisrv\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\solisrv\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [10]:
# Your specific data path
data_path = r"C:\Users\solisrv\OneDrive - beloit.edu\Desktop\MIND\MINDsmall_train"

# Load Data
news_cols = ['news_id', 'category', 'subcategory', 'title', 'abstract', 'url', 'title_entities', 'abstract_entities']
news_df = pd.read_csv(os.path.join(data_path, 'news.tsv'), sep='\t', names=news_cols)

beh_cols = ['impression_id', 'user_id', 'time', 'history', 'impressions']
behaviors_df = pd.read_csv(os.path.join(data_path, 'behaviors.tsv'), sep='\t', names=beh_cols)

print(f"Loaded {len(news_df)} news articles and {len(behaviors_df)} behaviors.")

Loaded 51282 news articles and 156965 behaviors.


In [11]:
class NewsTokenizer:
    def __init__(self, max_title_len=30, min_word_freq=2):
        self.max_title_len = max_title_len
        self.min_word_freq = min_word_freq
        self.word2idx = {'<PAD>': 0, '<UNK>': 1}

    def build_vocab(self, titles):
        word_counts = Counter()
        for title in titles:
            if pd.isna(title): continue
            tokens = nltk.word_tokenize(str(title).lower())
            word_counts.update(tokens)
        for word, count in word_counts.items():
            if count >= self.min_word_freq:
                self.word2idx[word] = len(self.word2idx)
        print(f'Vocabulary size: {len(self.word2idx)}')

    def encode_title(self, title):
        tokens = nltk.word_tokenize(str(title).lower())
        indices = [self.word2idx.get(t, 1) for t in tokens]
        if len(indices) < self.max_title_len:
            indices += [0] * (self.max_title_len - len(indices))
        else:
            indices = indices[:self.max_title_len]
        return indices

def load_glove(glove_path, word2idx, embed_dim=300):
    embedding_matrix = np.random.normal(
        size=(len(word2idx), embed_dim)).astype('float32') * 0.1
    embedding_matrix[0] = 0  # PAD vector
    found = 0
    
    # Fix: Added error handling and ensured the assignment is on one line
    if not os.path.exists(glove_path):
        print("GloVe file not found. Initializing with random embeddings.")
        return embedding_matrix

    with open(glove_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            word = parts[0]
            if word in word2idx:
                # Corrected syntax: assignment on the same line
                embedding_matrix[word2idx[word]] = np.array(parts[1:], dtype='float32')
                found += 1
    print(f'Found {found}/{len(word2idx)} words in GloVe')
    return embedding_matrix

In [12]:
# Initialize Tokenizer
tokenizer = NewsTokenizer()
tokenizer.build_vocab(news_df['title'])

# Map News IDs to encoded titles
news_encoded = {
    row['news_id']: tokenizer.encode_title(row['title']) 
    for _, row in news_df.iterrows()
}

def parse_behaviors(behaviors_df, news_encoded, neg_k=4):
    samples = []
    for _, row in behaviors_df.iterrows():
        history = row['history'].split() if pd.notna(row['history']) else []
        # Filter news that exists in our news_encoded dictionary
        history_encoded = [news_encoded[nid] for nid in history if nid in news_encoded]
        
        # Ensure we have at least some history
        if not history_encoded:
            history_encoded = [[0] * 30] 

        impressions = row['impressions'].split()
        pos = [imp.split('-')[0] for imp in impressions if imp.endswith('-1')]
        neg = [imp.split('-')[0] for imp in impressions if imp.endswith('-0')]
        
        for p in pos:
            if p not in news_encoded: continue
            
            # Negative sampling
            actual_neg = [n for n in neg if n in news_encoded]
            if len(actual_neg) < neg_k:
                sampled_neg = actual_neg + [list(news_encoded.keys())[0]] * (neg_k - len(actual_neg))
            else:
                sampled_neg = np.random.choice(actual_neg, size=neg_k, replace=False)
            
            candidates = [p] + list(sampled_neg)
            
            samples.append({
                'history': torch.tensor(history_encoded[-50:]),
                'candidates': torch.tensor([news_encoded[c] for c in candidates]),
                'label': 0 # The positive is always at index 0
            })
    return samples

train_samples = parse_behaviors(behaviors_df, news_encoded)

Vocabulary size: 20774


In [13]:
# Create DataLoader
# Note: You'll need a custom collate function if history lengths vary
train_loader = DataLoader(train_samples, batch_size=32, shuffle=True)

# Example Model Initialization (Assuming NRMS is defined)
# model = NRMS(embedding_matrix)
# optimizer = optim.Adam(model.parameters(), lr=0.001)
# criterion = nn.CrossEntropyLoss()

print("Ready for training.")

Ready for training.
